In [ ]:
# I used pythons 3.12.13
import time

import torch
from torch.utils.data import DataLoader
from torchvision import datasets as tvd

from common import REPO_ROOT, get_device
import open_clip

In [ ]:
CKPT_PATH = REPO_ROOT / "tulip-so400m-14-384.ckpt"
MODEL_NAME = "TULIP-so400m-14-384"
MODEL_RESOLUTION = 384

DATASET = "oxford_pet"
# DATASET = "food101"
# DATASET = "dtd"
# DATASET = "aircraft"
DATA_ROOT = REPO_ROOT / "final-project" / "data"
OUT_ROOT = REPO_ROOT / "final-project" / "features" / DATASET
OUT_ROOT.mkdir(parents=True, exist_ok=True)

BATCH_SIZE = 32
NUM_WORKERS = 4

DEVICE = get_device()

AVAILABLE_DATASETS = {
        "oxford_pet": { "dataset": tvd.OxfordIIITPet, "train":"trainval", "test":"test"},
        "food101": { "dataset": tvd.Food101, "train":"train", "test":"test"},
        "dtd": { "dataset": tvd.DTD, "train":"train", "test":"test"},
        "aircraft": { "dataset": tvd.FGVCAircraft, "train":"train", "test":"test"},
    }

In [ ]:
model, _, preprocess = open_clip.create_model_and_transforms(
    MODEL_NAME, pretrained=str(CKPT_PATH)
)
model.eval().to(DEVICE)
for param in model.parameters():
    param.requires_grad_(False)

with torch.no_grad():
    dummy = torch.zeros(1, 3, MODEL_RESOLUTION, MODEL_RESOLUTION, device=DEVICE)
    embed_dim = model.encode_image(dummy).shape[-1]
print("embedding dim:", embed_dim)

In [ ]:
@torch.no_grad()
def extract(split):
    dataset_info = AVAILABLE_DATASETS[DATASET]
    print(dataset_info)
    dataset = dataset_info['dataset'](
        root=DATA_ROOT, split=dataset_info[split], download=True, transform=preprocess
    )

    loader = DataLoader(
        dataset, batch_size=BATCH_SIZE, shuffle=False,
        num_workers=NUM_WORKERS, pin_memory=(DEVICE.type == "cuda"),
    )
    num_samples = len(dataset)
    features = torch.empty(num_samples, embed_dim, dtype=torch.float16)
    labels = torch.empty(num_samples, dtype=torch.int64)

    samples_written = 0
    start_time = time.time()
    for batch_index, (images, batch_labels) in enumerate(loader):
        images = images.to(DEVICE, non_blocking=True)
        batch_embeddings = model.encode_image(images).float()
        current_batch_size = batch_embeddings.shape[0]
        slice_end = samples_written + current_batch_size
        features[samples_written:slice_end] = batch_embeddings.cpu().to(torch.float16)
        labels[samples_written:slice_end] = batch_labels
        samples_written = slice_end
        if batch_index % 20 == 0:
            elapsed = time.time() - start_time
            print(f"  [{split}] {samples_written}/{num_samples}  ({elapsed:.1f}s)")

    total_seconds = time.time() - start_time
    print(f"  [{split}] done in {total_seconds:.1f}s — features {tuple(features.shape)}")
    return features, labels

In [ ]:
for split in ("train", "test"):
    features, labels = extract(split)
    torch.save(features, OUT_ROOT / f"{split}_features.pt")
    torch.save(labels, OUT_ROOT / f"{split}_labels.pt")
    print(f"saved to {OUT_ROOT}/{split}_*.pt")

In [ ]:
train_features = torch.load(OUT_ROOT / "train_features.pt")
train_labels = torch.load(OUT_ROOT / "train_labels.pt")
test_features = torch.load(OUT_ROOT / "test_features.pt")
test_labels = torch.load(OUT_ROOT / "test_labels.pt")

print("train:", train_features.shape, train_features.dtype, "labels:", train_labels.shape)
print("test: ", test_features.shape, test_features.dtype, "labels:", test_labels.shape)
print("num classes:", int(train_labels.max().item()) + 1)
print("feature norm:", train_features.float().norm(dim=-1).mean().item())

## TODO
 - Train a linear probe on the features.
    - Use the saved features and labels to train a linear probe
    - Evaluate performance
    - Compare to the original results

 - Reproduce the original results on a dataset from the paper.